# *Flaviviridae* Glycoprotein Phylogenetic Structure Tree Comparison

Using the AA and AlfaFold2 data from Misfud et al. "Mapping glcoprotein structure reveals *Flaviviridae* evolutionary history" we generated Trees using three approaches:

 1. **Standard Foldtree** - from AF2 3D structures
 2. **ProstT5 Foldtree** - using ProstT5 to convert AA sequences into structure 3di
 3. **ESMdi Foldtree** - using ESM3di to convert AA sequences into structure 3di

 We then compare the infered phylogenetic trees using custom monophyletic scoring ...


Find the pdb data at https://zenodo.org/records/11092288

/

TO DO 

explain why structure based phylogeny makes sense for these proteins, maybe add some identity scoring related stuff, look into original notebook...

Use this notebook to make the phylogeny plots, if we end up using them...


In [1]:
# imports ...

from Bio import Phylo
import re

### Sequence Identity Check

In [2]:
import numpy as np
import pandas as pd
from Bio import Align


def compute_identity(seq1, seq2):
    """Return global sequence identity as a percentage."""
    aligner = Align.PairwiseAligner()
    aligner.mode = "global"

    alignment = aligner.align(seq1, seq2)[0]
    aligned_seq1, aligned_seq2 = alignment.format().splitlines()[:2]

    matches = sum(
        aa1 == aa2
        for aa1, aa2 in zip(aligned_seq1, aligned_seq2)
        if aa1 != "-" and aa2 != "-"
    )
    comparable_positions = sum(
        aa1 != "-" and aa2 != "-"
        for aa1, aa2 in zip(aligned_seq1, aligned_seq2)
    )

    return 100 * matches / comparable_positions if comparable_positions else 0.0

def compute_standard_global_identity(seq1, seq2):
    """
    Computes global sequence identity using the standard EMBOSS Needle definition:
    Identity = Matches / Total Alignment Length (including all gaps)
    """
    aligner = Align.PairwiseAligner()
    aligner.mode = "global"
    
    # Simple substitution matrix for amino acids (BLOSUM62 is standard)
    alignment = aligner.align(seq1, seq2)[0]
    aligned_seq1, aligned_seq2 = alignment.format().splitlines()[:2]
    
    matches = sum(aa1 == aa2 for aa1, aa2 in zip(aligned_seq1, aligned_seq2) if aa1 != "-")
    alignment_length = len(aligned_seq1)  # Includes matches, mismatches, and gaps
    
    return (matches / alignment_length) * 100 if alignment_length > 0 else 0.0


def compute_sequence_identity_statistics(fasta_file):
    """Compute pairwise sequence identity statistics.

    Returns
    -------
    pandas.Series
        Descriptive statistics for all unique sequence pairs.
    """
    sequences = {}
    current_seq_id = None
    current_seq = []

    with open(fasta_file) as fasta:
        for line in fasta:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if current_seq_id is not None:
                    sequences[current_seq_id] = "".join(current_seq)

                current_seq_id = line[1:].strip()
                current_seq = []
            else:
                current_seq.append(line)

        if current_seq_id is not None:
            sequences[current_seq_id] = "".join(current_seq)

    seq_ids = list(sequences)
    identities = []

    for i in range(len(seq_ids)):
        for j in range(i + 1, len(seq_ids)):
            identities.append(
                compute_identity(
                    sequences[seq_ids[i]],
                    sequences[seq_ids[j]],
                )
            )

    if not identities:
        raise ValueError("At least two sequences are required.")

    return pd.Series(identities, name="sequence_identity_percent").describe()

In [ ]:
compute_sequence_identity_statistics("data/nucleoproteins/refolded_fullglyco_E_aa.fas")

In [5]:
compute_sequence_identity_statistics("data/aa_sequences/refolded_fullglyco_E_aa.fas")

count    30381.000000
mean        22.115318
std          5.510315
min         17.500000
25%         18.421053
50%         19.718310
75%         24.137931
max         63.636364
Name: sequence_identity_percent, dtype: float64

In [8]:
compute_sequence_identity_statistics("data/aa_sequences/refolded_fullglyco_E1_aa.fas")

count    17955.000000
mean        19.067406
std          1.435435
min         17.500000
25%         18.181818
50%         18.666667
75%         19.444444
max         40.000000
Name: sequence_identity_percent, dtype: float64

In [7]:
compute_sequence_identity_statistics("data/aa_sequences/refolded_fullglyco_E2_aa.fas")

count    17766.000000
mean        21.684832
std          4.532534
min         17.500000
25%         18.666667
50%         20.000000
75%         22.950820
max         60.869565
Name: sequence_identity_percent, dtype: float64

In [ ]:
# biased by gaps, should use different algo here that accounts for sequence length differences

compute_sequence_identity_statistics("data/aa_sequences/refolded_fullglyco_aa.fas")

count    195625.000000
mean         24.362209
std           6.390952
min          17.500000
25%          19.444444
50%          22.580645
75%          27.450980
max          63.636364
Name: sequence_identity_percent, dtype: float64

In [12]:
def score_monophyly(filename):
    """
    Calculate the fraction of internal nodes whose descendant leaves all
    belong to the same group.

    Leaf names must follow this format:
        ABCD_[unique_name]

    Parameters
    ----------
    filename : str
        Path to the input Newick file.

    Returns
    -------
    tuple
        (monophyletic_internal_nodes, all_internal_nodes, fraction)
    """
    tree = Phylo.read(filename, "newick")
    group_pattern = re.compile(r"^([A-Za-z0-9]+)_")

    def get_group(leaf):
        if not leaf.name:
            raise ValueError("Encountered a leaf without a name.")

        match = group_pattern.match(leaf.name)
        if not match:
            raise ValueError(
                f"Invalid leaf name {leaf.name!r}: expected a four-letter "
                "group code followed by an underscore."
            )

        return match.group(1)

    internal_nodes = tree.get_nonterminals()
    monophyletic_nodes = 0

    for node in internal_nodes:
        descendant_groups = {
            get_group(leaf) for leaf in node.get_terminals()
        }

        if len(descendant_groups) == 1:
            monophyletic_nodes += 1

    total_nodes = len(internal_nodes)
    fraction = monophyletic_nodes / total_nodes if total_nodes else 0.0

    #print("-" * 40)
    #print(f"Filename: {filename}")
    print(f"Monophyletic internal nodes: {monophyletic_nodes}")
    print(f"All internal nodes: {total_nodes}")
    print(f"Fraction of monophyletic internal nodes: {fraction:.6f}")
    print("-" * 40)

### 1 - Combined E/E1/E2 Phylogeny

In [7]:
%%bash
esm3di foldtree -i data/aa_sequences/refolded_fullglyco_aa.fas -o results -d esm3di_fullglyco

/users/fweikert/ESM3di/brief_communication/flavi_glyco/results
14:38:16 [INFO] Launching FoldTree Snakemake pipeline...


Config file config.yaml is extended by additional config specified via the command line.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 4
Rules claiming more threads will be scaled down.
Job stats:
job                         count
------------------------  -------
all                             1
build_esm3di_foldseek_db        1
foldseek2distmat                1
foldseek_allvall                1
mad_root_struct                 1
postprocess_tree                1
quicktree                       1
total                           7

Select jobs to execute...

[Tue Sep 15 14:38:18 2026]
rule build_esm3di_foldseek_db:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/data/aa_sequences/refolded_fullglyco_aa.fas
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/db, /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/db_h, /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/e

                                                 0   ...    11
0  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...  3505
1  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...  1005
2  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...   948
3  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...   909
4  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...   886

[5 rows x 12 columns]
                                                    query  ...  bits
0            FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...  3505
1            FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...  1005
2            FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...   948
3            FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...   909
4            FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...   886
...                                                   ...  ...   ...
391871  PLPV_Bat_pestivirus_BtSk-PestV-1GX2017_E2_724-...  ...     7
391872  PLPV_Bat_pestivirus_BtSk-PestV-1GX2017_

[Tue Sep 15 14:39:51 2026]
Finished job 4.
3 of 7 steps (43%) done
Select jobs to execute...

[Tue Sep 15 14:39:51 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/logs/esm3di_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=esm3di_fullglyco, model=esm3di
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda/45f84118cfc7ecacf5a2b628cf228


         /-HPHV_Hepacivirus_sp._E1_201-379_AlphaFold
        |
      /-|   /-HPPV_Goose_pegivirus_1_E1_33-220_AlphaFold
     |  |  |
     |   \-|      /-HPPV_Goose_pegivirus_2_E1_29-214_AlphaFold
     |     |   /-|
     |     |  |   \-HPPV_Montifringilla_taczanowskii_pegivirus_E1_28-222_AlphaFold
     |      \-|
     |        |   /-HPPV_Leucosticte_brandti_pegivirus_XZN_E1_28-217_AlphaFold
     |         \-|
     |            \-HPPV_Passer_montanus_pegivirus_GXN_E1_27-210_AlphaFold
     |
     |         /-PLLG_Hangzhou_ochthera_mantis_flavivirus_1_isolate_SYFY8_E_392-793_ESMFold
     |      /-|
     |     |   \-HPPV_Longquan_Rhinolophus_pearsonii_pegivirus_E1_67-245_AlphaFold
     |     |
     |   /-|      /-HPPV_Bat_pegivirus_PDB-34.1_E1_289-471_AlphaFold
     |  |  |   /-|
     |  |  |  |  |   /-HPPV_Pegivirus_G_E1_285-477_AlphaFold
   /-|  |  |  |   \-|
  |  |  |   \-|      \-HPPV_Tree_shrew_pegivirus_E1_161-347_AlphaFold
  |  |  |     |
  |  |  |     |   /-HPPV_Bat_pegivirus_PDB-1

[Tue Sep 15 14:39:54 2026]
Finished job 2.
5 of 7 steps (71%) done
Select jobs to execute...

[Tue Sep 15 14:39:54 2026]
rule mad_root_struct:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_struct_tree.PP.nwk
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_struct_tree.PP.nwk.rooted
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/logs/esm3di_mad_root_struct.log
    jobid: 1
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_struct_tree.PP.nwk.rooted; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco/esm3di_struct_tree.PP.nwk
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=esm3di_fullglyco, model=esm3di
    resources: tmpdir=/tmp

Activating conda environment: ../.sna

14:40:05 [INFO] FoldTree pipeline completed successfully. Results saved in: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results


In [8]:
%%bash
esm3di foldtree -i data/aa_sequences/refolded_fullglyco_aa.fas -o results -d prostt5_fullglyco --use-prostt5

/users/fweikert/ESM3di/brief_communication/flavi_glyco/results
14:40:09 [INFO] Launching FoldTree Snakemake pipeline...


Config file config.yaml is extended by additional config specified via the command line.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 4
Rules claiming more threads will be scaled down.
Job stats:
job                          count
-------------------------  -------
all                              1
build_prostt5_foldseek_db        1
download_prostt5_weights         1
foldseek2distmat                 1
foldseek_allvall                 1
mad_root_struct                  1
postprocess_tree                 1
quicktree                        1
total                            8

Select jobs to execute...

[Tue Sep 15 14:40:11 2026]
rule download_prostt5_weights:
    output: prostt5/prostt5_model.pt
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/logs/download_prostt5.log
    jobid: 7
    reason: Missing output files: prostt5/prostt5_model.pt
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda/ea782262

                                                  0   ...    11
0  FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  3877
1  FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1540
2  FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1214
3  FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1054
4  FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1036

[5 rows x 12 columns]
                                                    query  ...  bits
0       FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  3877
1       FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1540
2       FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1214
3       FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1054
4       FJAF_Sea_firefly_flavivirus_orf_1_E_289-775_Al...  ...  1036
...                                                   ...  ...   ...
391871  PLLG_Lampyris_noctiluca_flavivirus_1_isolate_1...  ...   -19
391872  PLLG_Lampyris_noctiluca_flaviviru

[Tue Sep 15 14:43:19 2026]
Finished job 4.
4 of 8 steps (50%) done
Select jobs to execute...

[Tue Sep 15 14:43:19 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/logs/prostt5_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco, model=prostt5
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda/45f84118cfc7ecacf


         /-HPHV_Guangxi_chinese_leopard_gecko_hepacivirus_E1_660-850_AlphaFold
      /-|
     |  |   /-HPHV_Hainan_oriental_leaf-toed_gecko_hepacivirus_E1_219-407_AlphaFold
     |   \-|
     |     |   /-HPHV_Yili_teratoscincus_roborowskii_hepacivirus_E1_672-861_AlphaFold
     |      \-|
     |         \-HPHV_Gecko_hepacivirus_E1_311-502_AlphaFold
     |
     |         /-HPHV_Chinese_broad-headed_pond_turtle_hepacivirus_E1_298-486_AlphaFold
     |      /-|
     |     |   \-HPHV_Chinese_softshell_turtle_hepacivirus_E1_301-487_AlphaFold
     |     |
     |     |      /-HPPV_Goose_pegivirus_1_E1_33-220_AlphaFold
     |     |     |
     |     |   /-|      /-HPPV_Goose_pegivirus_2_E1_29-214_AlphaFold
     |   /-|  |  |   /-|
     |  |  |  |  |  |   \-HPPV_Montifringilla_taczanowskii_pegivirus_E1_28-222_AlphaFold
     |  |  |  |   \-|
     |  |  |  |     |   /-HPPV_Leucosticte_brandti_pegivirus_XZN_E1_28-217_AlphaFold
     |  |  |  |      \-|
     |  |  |  |         \-HPPV_Passer_montanus_pe

[Tue Sep 15 14:43:31 2026]
Finished job 2.
6 of 8 steps (75%) done
Select jobs to execute...

[Tue Sep 15 14:43:31 2026]
rule mad_root_struct:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_struct_tree.PP.nwk
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_struct_tree.PP.nwk.rooted
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/logs/prostt5_mad_root_struct.log
    jobid: 1
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_struct_tree.PP.nwk.rooted; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco/prostt5_struct_tree.PP.nwk
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco, model=prostt5
    resources: tmpdir=/tmp

Activating conda environm

14:44:11 [INFO] FoldTree pipeline completed successfully. Results saved in: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results


In [15]:
print("ProstT5 - FoldTree")
score_monophyly("results/prostt5_fullglyco/prostt5_struct_tree.PP.nwk.rooted")
print("ESM3Di - FoldTree")
score_monophyly("results/esm3di_fullglyco/esm3di_struct_tree.PP.nwk.rooted")

ProstT5 - FoldTree
Monophyletic internal nodes: 523
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.836800
----------------------------------------
ESM3Di - FoldTree
Monophyletic internal nodes: 556
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.889600
----------------------------------------


### 2 - Separate Trees for E, E1, and E2

In [17]:
%%bash
esm3di foldtree -i data/aa_sequences/refolded_fullglyco_E_aa.fas -o results -d esm3di_fullglyco_E

esm3di foldtree -i data/aa_sequences/refolded_fullglyco_E_aa.fas -o results -d prostt5_fullglyco_E --use-prostt5

/users/fweikert/ESM3di/brief_communication/flavi_glyco/results
14:51:57 [INFO] Launching FoldTree Snakemake pipeline...


Config file config.yaml is extended by additional config specified via the command line.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 4
Rules claiming more threads will be scaled down.
Job stats:
job                         count
------------------------  -------
all                             1
build_esm3di_foldseek_db        1
foldseek2distmat                1
foldseek_allvall                1
mad_root_struct                 1
postprocess_tree                1
quicktree                       1
total                           7

Select jobs to execute...

[Tue Sep 15 14:52:02 2026]
rule build_esm3di_foldseek_db:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/data/aa_sequences/refolded_fullglyco_E_aa.fas
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E/db, /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E/db_h, /users/fweikert/ESM3di/brief_communication/flavi_glyco/res

                                                 0   ...    11
0  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...  3505
1  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...  1005
2  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...   948
3  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...   909
4  FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb  ...   886

[5 rows x 12 columns]
                                                   query  ...  bits
0           FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...  3505
1           FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...  1005
2           FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...   948
3           FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...   909
4           FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold  ...   886
...                                                  ...  ...   ...
61004  PLUN_Shayang_spider_virus_4_strain_SYZZ-1_E_18...  ...    -4
61005  PLUN_Shayang_spider_virus_4_strain_SYZZ-1_E_18..

[Tue Sep 15 14:52:38 2026]
Finished job 4.
3 of 7 steps (43%) done
Select jobs to execute...

[Tue Sep 15 14:52:38 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E/esm3di_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E/esm3di_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E/logs/esm3di_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E/esm3di_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E/esm3di_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=esm3di_fullglyco_E, model=esm3di
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda/45f84118cfc7ecacf


   /-FJMB_Cacipacore_virus_E_289-787_AlphaFold
  |
  |   /-FJMB_Japanese_encephalitis_virus_E_295-789_AlphaFold
  |--|
  |  |   /-FJMB_Usutu_virus_E_294-789_AlphaFold
  |   \-|
  |     |   /-FJMB_Alfuy_virus_strain_MRM3929_E_295-790_AlphaFold
  |      \-|
  |         \-FJMB_Murray_Valley_encephalitis_virus_E_294-788_AlphaFold
  |
--|      /-FJMB_Yaounde_virus_strain_Dak_Ar_Y276_E_291-791_AlphaFold
  |   /-|
  |  |  |   /-FJMB_Koutango_virus_isolate_PM148_E_293-787_AlphaFold
  |  |   \-|
  |  |     |   /-FJMB_West_Nile_virus_lineage_2_E_291-784_AlphaFold
  |  |      \-|
  |  |        |   /-FJMB_Kunjin_virus_clone_FLSDX_E_291-787_AlphaFold
  |  |         \-|
  |  |            \-FJMB_West_Nile_virus_lineage_1_E_291-787_AlphaFold
  |  |
  |  |      /-FJMB_St._Louis_encephalitis_virus_strain_Palenque-A770_E_321-816_AlphaFold
   \-|   /-|
     |  |   \-FJMB_Saint_Louis_encephalitis_virus_E_289-788_AlphaFold
     |  |
     |  |         /-FJMB_T_Ho_virus_strain_T_Ho-Mex07_E_287-783_AlphaFold


...

                                            0   ...    11
0  FJMB_Dengue_virus_4_E_280-774_AlphaFold.pdb  ...  3989
1  FJMB_Dengue_virus_4_E_280-774_AlphaFold.pdb  ...  2193
2  FJMB_Dengue_virus_4_E_280-774_AlphaFold.pdb  ...  2108
3  FJMB_Dengue_virus_4_E_280-774_AlphaFold.pdb  ...  2026
4  FJMB_Dengue_virus_4_E_280-774_AlphaFold.pdb  ...  1892

[5 rows x 12 columns]
                                                   query  ...  bits
0                FJMB_Dengue_virus_4_E_280-774_AlphaFold  ...  3989
1                FJMB_Dengue_virus_4_E_280-774_AlphaFold  ...  2193
2                FJMB_Dengue_virus_4_E_280-774_AlphaFold  ...  2108
3                FJMB_Dengue_virus_4_E_280-774_AlphaFold  ...  2026
4                FJMB_Dengue_virus_4_E_280-774_AlphaFold  ...  1892
...                                                  ...  ...   ...
61004  PLLG_Hangzhou_ochthera_mantis_flavivirus_1_iso...  ...     6
61005  PLLG_Hangzhou_ochthera_mantis_flavivirus_1_iso...  ...     5
61006  PLLG_Hangz

[Tue Sep 15 14:54:12 2026]
Finished job 4.
4 of 8 steps (50%) done
Select jobs to execute...

[Tue Sep 15 14:54:12 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/logs/prostt5_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco_E, model=prostt5
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda/45f84


   /-FJMB_Japanese_encephalitis_virus_E_295-789_AlphaFold
  |
  |   /-FJMB_Usutu_virus_E_294-789_AlphaFold
  |--|
  |  |   /-FJMB_Alfuy_virus_strain_MRM3929_E_295-790_AlphaFold
  |   \-|
--|      \-FJMB_Murray_Valley_encephalitis_virus_E_294-788_AlphaFold
  |
  |   /-FJMB_Cacipacore_virus_E_289-787_AlphaFold
  |  |
  |  |      /-FJMB_Koutango_virus_isolate_PM148_E_293-787_AlphaFold
  |  |   /-|
  |  |  |  |   /-FJMB_West_Nile_virus_lineage_2_E_291-784_AlphaFold
   \-|  |   \-|
     |  |     |   /-FJMB_Kunjin_virus_clone_FLSDX_E_291-787_AlphaFold
     |  |      \-|
     |  |         \-FJMB_West_Nile_virus_lineage_1_E_291-787_AlphaFold
     |  |
      \-|   /-FJMB_Yaounde_virus_strain_Dak_Ar_Y276_E_291-791_AlphaFold
        |  |
        |  |      /-FJMB_Saint_Louis_encephalitis_virus_E_289-788_AlphaFold
        |  |   /-|
        |  |  |   \-FJMB_St._Louis_encephalitis_virus_strain_Palenque-A770_E_321-816_AlphaFold
        |  |  |
        |  |  |         /-FJMB_T_Ho_virus_strain_T_Ho-Me

[Tue Sep 15 14:54:14 2026]
Finished job 2.
6 of 8 steps (75%) done
Select jobs to execute...

[Tue Sep 15 14:54:14 2026]
rule mad_root_struct:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_struct_tree.PP.nwk
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_struct_tree.PP.nwk.rooted
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/logs/prostt5_mad_root_struct.log
    jobid: 1
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_struct_tree.PP.nwk.rooted; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E/prostt5_struct_tree.PP.nwk
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco_E, model=prostt5
    resources: tmpdir=/tmp

Activating co

14:54:17 [INFO] FoldTree pipeline completed successfully. Results saved in: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results


In [22]:
print("E glycoprotein\n")

print("ProstT5 - FoldTree")
score_monophyly("results/prostt5_fullglyco_E/prostt5_struct_tree.PP.nwk.rooted")
print("ESM3Di - FoldTree")
score_monophyly("results/esm3di_fullglyco_E/esm3di_struct_tree.PP.nwk.rooted")

E glycoprotein

ProstT5 - FoldTree
Monophyletic internal nodes: 215
All internal nodes: 246
Fraction of monophyletic internal nodes: 0.873984
----------------------------------------
ESM3Di - FoldTree
Monophyletic internal nodes: 213
All internal nodes: 246
Fraction of monophyletic internal nodes: 0.865854
----------------------------------------


In [18]:
%%bash
esm3di foldtree -i data/aa_sequences/refolded_fullglyco_E1_aa.fas -o results -d esm3di_fullglyco_E1

esm3di foldtree -i data/aa_sequences/refolded_fullglyco_E1_aa.fas -o results -d prostt5_fullglyco_E1 --use-prostt5

/users/fweikert/ESM3di/brief_communication/flavi_glyco/results
14:54:21 [INFO] Launching FoldTree Snakemake pipeline...


Config file config.yaml is extended by additional config specified via the command line.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 4
Rules claiming more threads will be scaled down.
Job stats:
job                         count
------------------------  -------
all                             1
build_esm3di_foldseek_db        1
foldseek2distmat                1
foldseek_allvall                1
mad_root_struct                 1
postprocess_tree                1
quicktree                       1
total                           7

Select jobs to execute...

[Tue Sep 15 14:54:23 2026]
rule build_esm3di_foldseek_db:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/data/aa_sequences/refolded_fullglyco_E1_aa.fas
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E1/db, /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E1/db_h, /users/fweikert/ESM3di/brief_communication/flavi_glyco/

                                                  0   ...    11
0  HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFo...  ...  1159
1  HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFo...  ...   575
2  HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFo...  ...   476
3  HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFo...  ...   457
4  HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFo...  ...   312

[5 rows x 12 columns]
                                                  query  ...  bits
0      HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFold  ...  1159
1      HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFold  ...   575
2      HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFold  ...   476
3      HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFold  ...   457
4      HPHV_Bald_eagle_hepacivirus_E1_170-361_AlphaFold  ...   312
...                                                 ...  ...   ...
36095           HPHV_Hepacivirus_N_E1_151-347_AlphaFold  ...    32
36096           HPHV_Hepacivirus_N_E1_151-347_AlphaFold  

[Tue Sep 15 14:54:44 2026]
Finished job 4.
3 of 7 steps (43%) done
Select jobs to execute...

[Tue Sep 15 14:54:44 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E1/esm3di_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E1/esm3di_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E1/logs/esm3di_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E1/esm3di_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E1/esm3di_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=esm3di_fullglyco_E1, model=esm3di
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda/45f84118cfc


      /-HPHV_Wufeng_Typhlomys_cinereus_hepacivirus_1_E1_173-367_AlphaFold
   /-|
  |  |   /-HPHV_Hepacivirus_I_E1_180-369_AlphaFold
  |   \-|
  |      \-HPHV_Sigmodontinae_hepacivirus_E1_170-367_AlphaFold
  |
  |      /-HPHV_Ringtail_hepacivirus_E1_164-353_AlphaFold
  |   /-|
  |  |  |   /-HPHV_Guereza_hepacivirus_E1_180-392_AlphaFold
  |  |   \-|
  |  |      \-HPHV_Crab_eating_macaque_hepacivirus_E1_174-383_AlphaFold
  |--|
  |  |   /-HPHV_Possum_hepacivrus_E1_166-342_AlphaFold
  |  |  |
  |  |  |      /-HPHV_Sifaka_hepacivirus_MH824541_E1_200-394_AlphaFold
  |   \-|   /-|
  |     |  |   \-HPHV_Sifaka_hepacivirus_MT371438_E1_202-393_AlphaFold
  |     |  |
  |      \-|   /-HPHV_Hepacivirus_sp._clone_B32_E1_157-338_AlphaFold
  |        |  |
  |        |  |   /-HPHV_Norway_rat_hepacivirus_2_E1_164-382_AlphaFold
  |        |  |  |
  |         \-|  |      /-HPHV_Rodent_hepacivirus_CRT682COD2010_E1_173-368_AlphaFold
  |           |  |     |
  |           |  |   /-|      /-HPHV_Rodent_hepac

...

                                                  0   ...    11
0  HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...  1188
1  HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   670
2  HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   648
3  HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   648
4  HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   640

[5 rows x 12 columns]
                                                   query  ...  bits
0      HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...  1188
1      HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   670
2      HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   648
3      HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   648
4      HPHV_Rodent_hepacivirus_TA142TZA2013_E1_168-35...  ...   640
...                                                  ...  ...   ...
36095            HPHV_Hepacivirus_M_E1_191-382_AlphaFold  ...     0
36096            HPHV_Hepacivirus_M_E1_191-382_Al

[Tue Sep 15 14:56:04 2026]
Finished job 4.
4 of 8 steps (50%) done
Select jobs to execute...

[Tue Sep 15 14:56:04 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/logs/prostt5_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco_E1, model=prostt5
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda


   /-HPHV_Norway_rat_hepacivirus_2_E1_164-382_AlphaFold
  |
  |      /-HPHV_Hepacivirus_P_E1_148-338_AlphaFold
  |   /-|
  |  |  |   /-HPHV_Sifaka_hepacivirus_MH824541_E1_200-394_AlphaFold
  |  |   \-|
  |  |      \-HPHV_Sifaka_hepacivirus_MT371438_E1_202-393_AlphaFold
  |  |
  |  |   /-HPHV_Rodent_hepacivirus_CRT682COD2010_E1_173-368_AlphaFold
  |--|  |
  |  |  |         /-HPHV_Northern_treeshrew_hepacivirus_E1_155-346_AlphaFold
  |  |  |      /-|
  |  |  |     |  |   /-HPHV_Rodent_hepacivirus_TA085TZA2013_E1_223-407_AlphaFold
  |  |  |     |   \-|
  |  |  |     |     |   /-HPHV_Rodent_hepacivirus_TA498-CTZA2013_E1_251-435_AlphaFold
  |  |  |     |      \-|
  |   \-|     |        |   /-HPHV_Rodent_hepacivirus_CRT382COD2010_E1_252-436_AlphaFold
  |     |   /-|         \-|
  |     |  |  |            \-HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E1_252-435_AlphaFold
  |     |  |  |
  |     |  |  |      /-HPHV_Rodent_hepacivirus_rn-1_E1_246-450_AlphaFold
  |     |  |  |   /-|
  |     |  |  |

[Tue Sep 15 14:56:10 2026]
Finished job 2.
6 of 8 steps (75%) done
Select jobs to execute...

[Tue Sep 15 14:56:10 2026]
rule mad_root_struct:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_struct_tree.PP.nwk
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_struct_tree.PP.nwk.rooted
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/logs/prostt5_mad_root_struct.log
    jobid: 1
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_struct_tree.PP.nwk.rooted; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E1/prostt5_struct_tree.PP.nwk
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco_E1, model=prostt5
    resources: tmpdir=/tmp

Activat

14:56:12 [INFO] FoldTree pipeline completed successfully. Results saved in: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results


In [23]:
print("E1 glycoprotein\n")

print("ProstT5 - FoldTree")
score_monophyly("results/prostt5_fullglyco_E1/prostt5_struct_tree.PP.nwk.rooted")
print("ESM3Di - FoldTree")
score_monophyly("results/esm3di_fullglyco_E1/esm3di_struct_tree.PP.nwk.rooted")

E1 glycoprotein

ProstT5 - FoldTree
Monophyletic internal nodes: 177
All internal nodes: 189
Fraction of monophyletic internal nodes: 0.936508
----------------------------------------
ESM3Di - FoldTree
Monophyletic internal nodes: 180
All internal nodes: 189
Fraction of monophyletic internal nodes: 0.952381
----------------------------------------


In [19]:
%%bash
esm3di foldtree -i data/aa_sequences/refolded_fullglyco_E2_aa.fas -o results -d esm3di_fullglyco_E2

esm3di foldtree -i data/aa_sequences/refolded_fullglyco_E2_aa.fas -o results -d prostt5_fullglyco_E2 --use-prostt5

/users/fweikert/ESM3di/brief_communication/flavi_glyco/results
14:56:17 [INFO] Launching FoldTree Snakemake pipeline...


Config file config.yaml is extended by additional config specified via the command line.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 4
Rules claiming more threads will be scaled down.
Job stats:
job                         count
------------------------  -------
all                             1
build_esm3di_foldseek_db        1
foldseek2distmat                1
foldseek_allvall                1
mad_root_struct                 1
postprocess_tree                1
quicktree                       1
total                           7

Select jobs to execute...

[Tue Sep 15 14:56:19 2026]
rule build_esm3di_foldseek_db:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/data/aa_sequences/refolded_fullglyco_E2_aa.fas
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E2/db, /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E2/db_h, /users/fweikert/ESM3di/brief_communication/flavi_glyco/

                                                  0   ...    11
0  HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFo...  ...  1929
1  HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFo...  ...   338
2  HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFo...  ...   334
3  HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFo...  ...   320
4  HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFo...  ...   220

[5 rows x 12 columns]
                                                   query  ...  bits
0       HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFold  ...  1929
1       HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFold  ...   338
2       HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFold  ...   334
3       HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFold  ...   320
4       HPHV_Bald_eagle_hepacivirus_E2_364-657_AlphaFold  ...   220
...                                                  ...  ...   ...
35716  PLPV_Rodent_pestivirus_isolate_RtNn-PestVHuB20...  ...    19
35717  PLPV_Rodent_pestivirus_isolate_RtNn-PestVH

[Tue Sep 15 14:56:42 2026]
Finished job 4.
3 of 7 steps (43%) done
Select jobs to execute...

[Tue Sep 15 14:56:42 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E2/esm3di_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E2/esm3di_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E2/logs/esm3di_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E2/esm3di_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/esm3di_fullglyco_E2/esm3di_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=esm3di_fullglyco_E2, model=esm3di
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda/45f84118cfc


      /-HPHV_Ringtail_hepacivirus_E2_356-639_AlphaFold
     |
   /-|      /-HPHV_Rodent_hepacivirus_CRT682COD2010_E2_390-643_AlphaFold
  |  |   /-|
  |  |  |   \-HPHV_Northern_treeshrew_hepacivirus_E2_369-627_AlphaFold
  |   \-|
  |     |   /-HPHV_Hepacivirus_sp._clone_B32_E2_340-608_AlphaFold
  |      \-|
  |         \-HPHV_Hepacivirus_P_E2_339-606_AlphaFold
  |
  |         /-HPHV_Sifaka_hepacivirus_MT371438_E2_397-657_AlphaFold
  |      /-|
  |     |   \-HPHV_Sifaka_hepacivirus_MH824541_E2_405-654_AlphaFold
  |   /-|
  |  |  |   /-HPHV_Norway_rat_hepacivirus_2_E2_383-652_AlphaFold
  |  |   \-|
  |  |      \-HPHV_Possum_hepacivrus_E2_342-627_AlphaFold
  |  |
  |--|      /-HPHV_Rodent_hepacivirus_RO22799BeAn812052_E2_357-619_AlphaFold
  |  |   /-|
  |  |  |   \-HPHV_Rodent_hepacivirus_B349PAN2014_E2_352-625_AlphaFold
  |  |  |
  |  |  |      /-HPHV_Rodent_hepacivirus_05VZ-14-118_E2_358-619_AlphaFold
  |   \-|   /-|
  |     |  |  |   /-HPHV_Rhizomys_pruinosus_hepacivirus_E2_363-635_Alp

...

                                                  0   ...    11
0  HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1876
1  HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1474
2  HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1471
3  HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1378
4  HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...   610

[5 rows x 12 columns]
                                                   query  ...  bits
0      HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1876
1      HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1474
2      HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1471
3      HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...  1378
4      HPHV_Rodent_hepacivirus_MOZ329-CMOZ2011_E2_439...  ...   610
...                                                  ...  ...   ...
35716  HPPV_Wufeng_Niviventer_fulvescens_pegivirus_1_...  ...     4
35717  HPPV_Wufeng_Niviventer_fulvescens_pegiviru

[Tue Sep 15 14:58:00 2026]
Finished job 4.
4 of 8 steps (50%) done
Select jobs to execute...

[Tue Sep 15 14:58:00 2026]
rule quicktree:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_fastmemat.txt
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_struct_tree.nwk
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/logs/prostt5_quicktree.log
    jobid: 3
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_struct_tree.nwk; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_fastmemat.txt
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco_E2, model=prostt5
    resources: tmpdir=/tmp

Activating conda environment: ../.snakemake_conda


         /-HPHV_Ringtail_hepacivirus_E2_356-639_AlphaFold
      /-|
     |  |   /-HPHV_Hepacivirus_P_E2_339-606_AlphaFold
     |   \-|
     |      \-HPHV_Hepacivirus_sp._clone_B32_E2_340-608_AlphaFold
   /-|
  |  |      /-HPHV_Sifaka_hepacivirus_MT371438_E2_397-657_AlphaFold
  |  |   /-|
  |  |  |   \-HPHV_Sifaka_hepacivirus_MH824541_E2_405-654_AlphaFold
  |   \-|
  |     |   /-HPHV_Northern_treeshrew_hepacivirus_E2_369-627_AlphaFold
  |      \-|
  |         \-HPHV_Rodent_hepacivirus_CRT682COD2010_E2_390-643_AlphaFold
  |
  |   /-HPHV_Norway_rat_hepacivirus_2_E2_383-652_AlphaFold
  |  |
  |  |         /-HPHV_Gerbil_hepacivirus_E2_407-679_AlphaFold
  |  |      /-|
  |--|     |   \-HPHV_Wufeng_Typhlomys_cinereus_hepacivirus_2_E2_363-626_AlphaFold
  |  |   /-|
  |  |  |  |   /-HPHV_Rodent_hepacivirus_RO22799BeAn812052_E2_357-619_AlphaFold
  |  |  |   \-|
  |  |  |      \-HPHV_Rodent_hepacivirus_B349PAN2014_E2_352-625_AlphaFold
  |   \-|
  |     |   /-HPHV_Rodent_hepacivirus_TZ25757TZA201

[Tue Sep 15 14:58:02 2026]
Finished job 2.
6 of 8 steps (75%) done
Select jobs to execute...

[Tue Sep 15 14:58:02 2026]
rule mad_root_struct:
    input: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_struct_tree.PP.nwk
    output: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_struct_tree.PP.nwk.rooted
    log: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/logs/prostt5_mad_root_struct.log
    jobid: 1
    reason: Missing output files: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_struct_tree.PP.nwk.rooted; Input files updated by another job: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results/prostt5_fullglyco_E2/prostt5_struct_tree.PP.nwk
    wildcards: out_dir=/users/fweikert/ESM3di/brief_communication/flavi_glyco/results, dataset=prostt5_fullglyco_E2, model=prostt5
    resources: tmpdir=/tmp

Activat

14:58:04 [INFO] FoldTree pipeline completed successfully. Results saved in: /users/fweikert/ESM3di/brief_communication/flavi_glyco/results


In [24]:
print("E2 glycoprotein\n")

print("ProstT5 - FoldTree")
score_monophyly("results/prostt5_fullglyco_E2/prostt5_struct_tree.PP.nwk.rooted")
print("ESM3Di - FoldTree")
score_monophyly("results/esm3di_fullglyco_E2/esm3di_struct_tree.PP.nwk.rooted")

E2 glycoprotein

ProstT5 - FoldTree
Monophyletic internal nodes: 181
All internal nodes: 188
Fraction of monophyletic internal nodes: 0.962766
----------------------------------------
ESM3Di - FoldTree
Monophyletic internal nodes: 180
All internal nodes: 188
Fraction of monophyletic internal nodes: 0.957447
----------------------------------------
